In [6]:
import pandas as pd
import altair as alt
import ast

nih_merged2_with_traits = pd.read_csv('nih_merged2_with_traits.csv')

### 1. Who was the funding for, and what do those groups do?

In [7]:
terminated = nih_merged2_with_traits[nih_merged2_with_traits['has_been_terminated'] == True]

words = terminated['flagged_words'].dropna().str.split(',').explode().str.strip()
top_words = words.value_counts().head(15).reset_index()
top_words.columns = ['word', 'count']

chart = alt.Chart(top_words).mark_bar().encode(
    x=alt.X('count:Q', title='Number of Terminated Grants'),
    y=alt.Y('word:N', sort='-x', title='Flagged Term'),
    tooltip=['word', 'count']
)
chart

alt.Chart(...)

The most common flagged terms among terminated grants are things like "trans," "diverse," "gender," "minority," "race," "women," "sex," and "underrepresented." This means a large share of the funding was for research on or by LGBTQ+ populations, racial/ethnic minorities, women's health, and broader diversity/health-disparities work. These are typically epidemiological, public health, and biomedical studies aimed at understanding health disparities, disease risk, or treatment outcomes in these populations — not advocacy, but research about them (e.g., studying minority stress and substance use, or sex differences in disease).

### 2. Did the cuts disproportionately affect certain groups or institutions?

In [8]:
traits = ['Public','Private','R1','R2','Land grant','Hispanic Serving',
          'Tribal/Native Serving','HBCU/Predominantly Black','Research','Rural']

rows = []
for t in traits:
    sub = nih_merged2_with_traits[nih_merged2_with_traits[t] == 1]
    rows.append({'trait': t, 'n_grants': len(sub), 'termination_rate': sub['has_been_terminated'].mean()})

trait_term_df = pd.DataFrame(rows).sort_values('termination_rate', ascending=False)

overall_rate = nih_merged2_with_traits['has_been_terminated'].mean()
print(f"Overall termination rate: {overall_rate:.1%}")
print(trait_term_df)

chart = alt.Chart(trait_term_df).mark_bar().encode(
    x=alt.X('termination_rate:Q', axis=alt.Axis(format='%'), title='Termination Rate'),
    y=alt.Y('trait:N', sort='-x', title='Institution / Grant Trait'),
    tooltip=['trait', 'n_grants', alt.Tooltip('termination_rate:Q', format='.1%')]
)
chart

Overall termination rate: 59.5%
                      trait  n_grants  termination_rate
9                     Rural         3          1.000000
5          Hispanic Serving       237          0.983122
7  HBCU/Predominantly Black        35          0.971429
8                  Research        79          0.962025
6     Tribal/Native Serving       890          0.949438
0                    Public      1676          0.938544
3                        R2       174          0.931034
4                Land grant      1136          0.779049
2                        R1      4992          0.541867
1                   Private      3688          0.401030


alt.Chart(...)

Minority-serving institutions (Hispanic Serving, HBCU, Tribal/Native Serving) and public institutions were terminated at roughly double the rate of private institutions overall.

### 3. Public vs. private grant termination

In [9]:
def classify(row):
    if row['Public'] == 1:
        return 'Public'
    elif row['Private'] == 1:
        return 'Private'
    return 'Other/Unknown'

nih_merged2_with_traits['org_sector'] = nih_merged2_with_traits.apply(classify, axis=1)

sector_status = nih_merged2_with_traits[nih_merged2_with_traits['org_sector'] != 'Other/Unknown'].groupby(
    ['org_sector', 'detailed_status']
).size().reset_index(name='count')

sector_totals = nih_merged2_with_traits.groupby('org_sector').size()
sector_status['pct'] = sector_status.apply(lambda r: r['count'] / sector_totals[r['org_sector']], axis=1)

chart = alt.Chart(sector_status).mark_bar().encode(
    x=alt.X('pct:Q', axis=alt.Axis(format='%'), title='Share of Grants', stack='normalize'),
    y=alt.Y('org_sector:N', title='Sector'),
    color=alt.Color('detailed_status:N', title='Status'),
    tooltip=['org_sector', 'detailed_status', 'count', alt.Tooltip('pct:Q', format='.1%')]
)
chart

alt.Chart(...)

Public institutions were terminated at 93.9% vs. 40.1% for private institutions — more than double. But this comes with an important caveat: one important confound is that several specific elite private universities (Harvard especially) were targeted independently of their private status — Harvard Medical School, Harvard School of Public Health, and Harvard University combined account for over 600 terminated grants on their own, which pulls the "private" termination count up despite private institutions having a lower overall rate. So the public/private gap isn't purely about ownership type — it's a mix of broad public-institution impact plus concentrated cuts at a handful of specific private institutions.

### 4. Which institutions and centers had the most cuts?

In [10]:
# Top institutions
top_orgs = terminated['org_name'].value_counts().head(15).reset_index()
top_orgs.columns = ['org_name', 'terminated_count']

chart_orgs = alt.Chart(top_orgs).mark_bar().encode(
    x=alt.X('terminated_count:Q', title='Number of Terminated Grants'),
    y=alt.Y('org_name:N', sort='-x', title='Institution'),
    tooltip=['org_name', 'terminated_count']
)
chart_orgs

alt.Chart(...)

In [11]:
# Top NIH institutes/centers — agency_ic_admin is a stringified dict, so parse it
def get_abbr(x):
    try:
        return ast.literal_eval(x).get('abbreviation')
    except Exception:
        return None

terminated_copy = terminated.copy()
terminated_copy['ic_abbr'] = terminated_copy['agency_ic_admin'].apply(get_abbr)

top_ics = terminated_copy['ic_abbr'].value_counts().head(10).reset_index()
top_ics.columns = ['institute', 'terminated_count']

chart_ics = alt.Chart(top_ics).mark_bar().encode(
    x=alt.X('terminated_count:Q', title='Number of Terminated Grants'),
    y=alt.Y('institute:N', sort='-x', title='NIH Institute/Center'),
    tooltip=['institute', 'terminated_count']
)
chart_ics

alt.Chart(...)

By institution: UCLA (477), Harvard Medical School (320), Columbia University Health Sciences (164), Harvard School of Public Health (135), and Harvard University (130) lead — note the three Harvard-affiliated entries would total ~590 if combined, making Harvard the single largest target by far.
By NIH institute/center: NIGMS (501), NIAID (393), NIMH (325), NCI (263), and NIA (258) had the most terminations — though this partly tracks each institute's overall grant volume rather than a targeted rate (NIGMS and NIAID are simply large institutes).

### 5. Is there a trend in the cuts over time?

In [47]:
nih_merged2_with_traits['first_termination_date'] = pd.to_datetime(
    nih_merged2_with_traits['first_termination_date'], errors='coerce'
)

trend = nih_merged2_with_traits.dropna(subset=['first_termination_date']).copy()
trend['month'] = trend['first_termination_date'].dt.to_period('M').dt.to_timestamp()
monthly_counts = trend.groupby('month').size().reset_index(name='terminations')

chart_trend = alt.Chart(monthly_counts).mark_line(point=True).encode(
    x=alt.X('month:T', title='Month'),
    y=alt.Y('terminations:Q', title='Number of Terminations'),
    tooltip=['month', 'terminations']
).properties(
    width=600
).interactive()
chart_trend

alt.Chart(...)

There's a sharp, concentrated spike rather than a steady trend: terminations jumped from near-zero in early 2025 to 764 in March, 694 in April, and peaked at 1,045 in May 2025, then fell off through summer (346 in June, 490 in July) and dropped to near-zero by fall 2025. This points to a short, intense termination wave in spring 2025 rather than a gradual or ongoing policy.

### Hypothesis 3 (Teresa): Grants are being cut disproportionately in certain areas like the Northeast

**What's informative about this view:** The Northeast does have the highest number of terminated grants (1,356, or 39.6% of all terminations nationwide) — more than any other region. This is important as it's where the most NIH-funded research happens overall (Harvard, Columbia, Yale, Penn, etc. are concentrated there).

**What could be improved about this view:** Looking at the rate rather than the raw count tells a different story, the Northeast actually has the lowest termination rate of any region (48.3%), well below the South (95.0%) and West (92.6%). The Northeast has so many total grants that even a below-average rate produces a large absolute number. So "disproportionate" doesn't hold up once we control for how much funding a region had to begin with.
Digging one level further, even the Northeast's 48.3% rate is being pulled up almost entirely by one institution (Harvard).

In [16]:
northeast = {'CT','ME','MA','NH','RI','VT','NJ','NY','PA'}
midwest = {'IL','IN','MI','OH','WI','IA','KS','MN','MO','NE','ND','SD'}
south = {'DE','FL','GA','MD','NC','SC','VA','DC','WV','AL','KY','MS','TN','AR','LA','OK','TX'}
west = {'AZ','CO','ID','MT','NV','NM','UT','WY','AK','CA','HI','OR','WA'}

def region(state):
    if state in northeast: return 'Northeast'
    if state in midwest: return 'Midwest'
    if state in south: return 'South'
    if state in west: return 'West'
    return 'Other/Territory'

nih_merged2_with_traits['region'] = nih_merged2_with_traits['org_state'].apply(region)
regional = nih_merged2_with_traits[nih_merged2_with_traits['region'] != 'Other/Territory']

reg_stats = regional.groupby('region')['has_been_terminated'].agg(
    n_grants='count', n_terminated='sum', rate='mean'
).reset_index()

print(reg_stats)

      region  n_grants  n_terminated      rate
0    Midwest      1096           338  0.308394
1  Northeast      2810          1356  0.482562
2      South       721           685  0.950069
3       West      1102          1020  0.925590


In [48]:
chart_rate = alt.Chart(reg_stats).mark_bar().encode(
    x=alt.X('rate:Q', axis=alt.Axis(format='%'), title='Termination Rate'),
    y=alt.Y('region:N', sort='-x', title='Region'),
    tooltip=['region', 'n_grants', 'n_terminated', alt.Tooltip('rate:Q', format='.1%')]
).properties(
    title='Region Termination Rates'
)
chart_rate

alt.Chart(...)

In [49]:
chart_count = alt.Chart(reg_stats).mark_bar(color='#c0392b').encode(
    x=alt.X('n_terminated:Q', title='Number of Terminated Grants'),
    y=alt.Y('region:N', sort='-x', title='Region'),
    tooltip=['region', 'n_terminated']
).properties(
    title='Region Termination Count'
)
chart_count

alt.Chart(...)

In [19]:
ne = regional[regional['region'] == 'Northeast']
harvard_mask = ne['org_name'].str.contains('HARVARD', na=False)

print('Harvard-affiliated:', harvard_mask.sum(), 'grants, rate:',
      f"{ne[harvard_mask]['has_been_terminated'].mean():.1%}")
print('Rest of Northeast:', (~harvard_mask).sum(), 'grants, rate:',
      f"{ne[~harvard_mask]['has_been_terminated'].mean():.1%}")

Harvard-affiliated: 684 grants, rate: 92.7%
Rest of Northeast: 2126 grants, rate: 34.0%


Excluding Harvard, the rest of the Northeast's termination rate drops to 34% which is below the national average of 59.5%, and well below the Midwest (30.8%, the actual lowest).

The data doesn't really support "Northeast" as a meaningful geographic pattern. The reason behind this regional effect is mostly because of 

1. the Northeast simply having more total NIH funding, and 

2. Harvard was being targeted specifically — not the region as a whole. 

The South and West show far higher termination rates, which is arguably the more relevant finding if the question is "where are cuts disproportionate," not just "where are there a lot of cuts."